MINI PROJECT - Praktikum 10: Boosting, Voting & Mini Project
Dataset: heart.csv (Heart Failure Prediction Dataset)


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    AdaBoostClassifier, GradientBoostingClassifier,
    RandomForestClassifier, VotingClassifier
)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix



===========================================================
1. LOAD & PREPROCESSING DATA
===========================================================


In [ ]:
df = pd.read_csv("/mnt/user-data/uploads/heart.csv")
print("Shape data:", df.shape)
print(df.head())
print("\nInfo data:")
print(df.info())
print("\nMissing values:\n", df.isnull().sum())
print("\nDistribusi target (HeartDisease):\n", df['HeartDisease'].value_counts())

# Encoding kolom kategorikal
cat_cols = df.select_dtypes(include='object').columns.tolist()
print("\nKolom kategorikal:", cat_cols)

df_encoded = df.copy()
le = LabelEncoder()
for col in cat_cols:
    df_encoded[col] = le.fit_transform(df_encoded[col])

X = df_encoded.drop("HeartDisease", axis=1)
y = df_encoded["HeartDisease"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {}



===========================================================
EKSPERIMEN 1: Logistic Regression (baseline)
===========================================================


In [ ]:
log = LogisticRegression(max_iter=5000, random_state=42)
log.fit(X_train_scaled, y_train)
acc_log = accuracy_score(y_test, log.predict(X_test_scaled))
cv_log = cross_val_score(log, X_train_scaled, y_train, cv=cv).mean()
results["Logistic Regression"] = {"Test Accuracy": acc_log, "CV Accuracy": cv_log}
print(f"\n[1] Logistic Regression -> Test Acc: {acc_log:.4f} | CV Acc: {cv_log:.4f}")



===========================================================
EKSPERIMEN 2: Random Forest (Ensemble - Bagging)
===========================================================


In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)
acc_rf = accuracy_score(y_test, rf.predict(X_test_scaled))
cv_rf = cross_val_score(rf, X_train_scaled, y_train, cv=cv).mean()
results["Random Forest"] = {"Test Accuracy": acc_rf, "CV Accuracy": cv_rf}
print(f"[2] Random Forest -> Test Acc: {acc_rf:.4f} | CV Acc: {cv_rf:.4f}")



===========================================================
EKSPERIMEN 3: AdaBoost (Ensemble - Boosting)
===========================================================


In [ ]:
ada = AdaBoostClassifier(n_estimators=100, random_state=42)
ada.fit(X_train_scaled, y_train)
acc_ada = accuracy_score(y_test, ada.predict(X_test_scaled))
cv_ada = cross_val_score(ada, X_train_scaled, y_train, cv=cv).mean()
results["AdaBoost"] = {"Test Accuracy": acc_ada, "CV Accuracy": cv_ada}
print(f"[3] AdaBoost -> Test Acc: {acc_ada:.4f} | CV Acc: {cv_ada:.4f}")



===========================================================
EKSPERIMEN 4: Gradient Boosting (Ensemble - Boosting)
===========================================================


In [ ]:
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train_scaled, y_train)
acc_gb = accuracy_score(y_test, gb.predict(X_test_scaled))
cv_gb = cross_val_score(gb, X_train_scaled, y_train, cv=cv).mean()
results["Gradient Boosting"] = {"Test Accuracy": acc_gb, "CV Accuracy": cv_gb}
print(f"[4] Gradient Boosting -> Test Acc: {acc_gb:.4f} | CV Acc: {cv_gb:.4f}")



===========================================================
EKSPERIMEN 5: Voting Classifier (Soft Voting)
===========================================================


In [ ]:
log_v = LogisticRegression(max_iter=5000, random_state=42)
rf_v = RandomForestClassifier(n_estimators=100, random_state=42)
gb_v = GradientBoostingClassifier(n_estimators=100, random_state=42)

voting = VotingClassifier(
    estimators=[('log', log_v), ('rf', rf_v), ('gb', gb_v)],
    voting='soft'
)
voting.fit(X_train_scaled, y_train)
acc_vote = accuracy_score(y_test, voting.predict(X_test_scaled))
cv_vote = cross_val_score(voting, X_train_scaled, y_train, cv=cv).mean()
results["Voting Classifier"] = {"Test Accuracy": acc_vote, "CV Accuracy": cv_vote}
print(f"[5] Voting Classifier -> Test Acc: {acc_vote:.4f} | CV Acc: {cv_vote:.4f}")



===========================================================
EKSPERIMEN 6: Hyperparameter Tuning (GridSearchCV - Random Forest)
===========================================================


In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)
grid.fit(X_train_scaled, y_train)

best_rf = grid.best_estimator_
acc_best_rf = accuracy_score(y_test, best_rf.predict(X_test_scaled))
results["Best Tuned Model (RF GridSearch)"] = {
    "Test Accuracy": acc_best_rf, "CV Accuracy": grid.best_score_
}
print(f"\n[6] Best Tuned RF Params: {grid.best_params_}")
print(f"    Best CV Score: {grid.best_score_:.4f} | Test Acc: {acc_best_rf:.4f}")



===========================================================
EKSPERIMEN 7: PCA + Gradient Boosting
===========================================================


In [ ]:
pca = PCA(n_components=8, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"\nVarians yang dijelaskan oleh 8 komponen PCA: "
      f"{pca.explained_variance_ratio_.sum():.4f}")

gb_pca = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb_pca.fit(X_train_pca, y_train)
acc_gb_pca = accuracy_score(y_test, gb_pca.predict(X_test_pca))
cv_gb_pca = cross_val_score(gb_pca, X_train_pca, y_train, cv=cv).mean()
results["Gradient Boosting + PCA"] = {"Test Accuracy": acc_gb_pca, "CV Accuracy": cv_gb_pca}
print(f"[7] Gradient Boosting + PCA -> Test Acc: {acc_gb_pca:.4f} | CV Acc: {cv_gb_pca:.4f}")



===========================================================
TABEL PERBANDINGAN AKHIR
===========================================================


In [ ]:
df_results = pd.DataFrame(results).T
df_results = df_results.sort_values("Test Accuracy", ascending=False)
print("\n================= TABEL PERBANDINGAN AKHIR =================")
print(df_results.round(4))
df_results.round(4).to_csv("/mnt/user-data/outputs/tabel_perbandingan.csv")



===========================================================
VISUALISASI
===========================================================
1. Bar chart perbandingan akurasi


In [ ]:
plt.figure(figsize=(10, 6))
df_plot = df_results.reset_index()
x = np.arange(len(df_plot))
width = 0.35
plt.bar(x - width/2, df_plot["Test Accuracy"], width, label="Test Accuracy")
plt.bar(x + width/2, df_plot["CV Accuracy"], width, label="CV Accuracy")
plt.xticks(x, df_plot["index"], rotation=30, ha="right")
plt.ylabel("Accuracy")
plt.title("Perbandingan Akurasi Model")
plt.ylim(0.5, 1.0)
plt.legend()
plt.tight_layout()
plt.savefig("/mnt/user-data/outputs/perbandingan_akurasi.png", dpi=150)
plt.close()

# 2. PCA explained variance plot
pca_full = PCA(random_state=42).fit(X_train_scaled)
plt.figure(figsize=(8, 5))
plt.plot(np.cumsum(pca_full.explained_variance_ratio_), marker='o')
plt.axhline(y=0.95, color='r', linestyle='--', label="95% variance")
plt.xlabel("Jumlah Komponen Utama")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA - Cumulative Explained Variance")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/mnt/user-data/outputs/pca_variance.png", dpi=150)
plt.close()

# 3. Confusion Matrix - Best Model
best_model_name = df_results.index[0]
print(f"\nModel terbaik berdasarkan Test Accuracy: {best_model_name}")

model_map = {
    "Logistic Regression": (log, X_test_scaled),
    "Random Forest": (rf, X_test_scaled),
    "AdaBoost": (ada, X_test_scaled),
    "Gradient Boosting": (gb, X_test_scaled),
    "Voting Classifier": (voting, X_test_scaled),
    "Best Tuned Model (RF GridSearch)": (best_rf, X_test_scaled),
    "Gradient Boosting + PCA": (gb_pca, X_test_pca),
}
best_model, X_test_for_best = model_map[best_model_name]
y_pred_best = best_model.predict(X_test_for_best)

cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Disease', 'Disease'],
            yticklabels=['No Disease', 'Disease'])
plt.title(f"Confusion Matrix - {best_model_name}")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.tight_layout()
plt.savefig("/mnt/user-data/outputs/confusion_matrix.png", dpi=150)
plt.close()

# 4. Feature importance (Random Forest)
plt.figure(figsize=(8, 6))
importances = rf.feature_importances_
feat_names = X.columns
idx_sort = np.argsort(importances)[::-1]
plt.barh(np.array(feat_names)[idx_sort][:10][::-1], importances[idx_sort][:10][::-1])
plt.title("Top 10 Feature Importance (Random Forest)")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig("/mnt/user-data/outputs/feature_importance.png", dpi=150)
plt.close()

print("\nClassification Report - Model Terbaik:")
print(classification_report(y_test, y_pred_best, target_names=['No Disease', 'Disease']))

print("\nSemua eksperimen selesai. File output disimpan di /mnt/user-data/outputs/")
